# E3 BioBART Sentence With-context Fine-tuning

This experiment keeps the BioBART training and evaluation pipeline unchanged and modifies only the encoder input. Each current sentence is accompanied by its immediate previous and next complex sentences from the same `pair_id`.


## 1.Imports and Configuration


In [ ]:
from __future__ import annotations

import ast
import gc
import os
import random
import re
import time
from collections import Counter
from pathlib import Path
from typing import Any

LOCAL_CACHE_DIR = Path.cwd() / ".cache"
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(LOCAL_CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(LOCAL_CACHE_DIR))
os.environ.setdefault("HF_HOME", str(LOCAL_CACHE_DIR / "huggingface"))
# Disable hf_transfer unless the package is explicitly installed.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import numpy as np
import pandas as pd
import sacrebleu
import torch
from bert_score import score as bert_score
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

try:
    import evaluate
except ImportError:
    evaluate = None

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())




In [ ]:
SEED = 42
MODEL_NAME = "GanjinZero/biobart-base"
MODEL_CANDIDATES = [
    MODEL_NAME,
    "facebook/bart-base",
]

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "sentence" / "raw"
TRAIN_PATH = DATA_DIR / "cochraneauto_sents_train.csv"
VAL_PATH = DATA_DIR / "cochraneauto_sents_val.csv"
TEST_PATH = DATA_DIR / "cochraneauto_sents_test.csv"

OUTPUT_DIR = PROJECT_ROOT / "models" / "biobart_sentence_context"
RESULTS_DIR = PROJECT_ROOT / "results"
PREDICTION_PATH = RESULTS_DIR / "biobart_sentence_context_predictions.csv"
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bf16_supported = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
fp16_enabled = False
bf16_enabled = bf16_supported

print("Parameters are fixed and logged.")


## 2.Dataset


In [ ]:
REQUIRED_COLUMNS = ["pair_id", "para_id", "sent_id", "label", "complex", "simple"]
KEEP_LABELS = {"rephrase", "ignore", "split"}
NO_PREVIOUS = "<NO PREVIOUS SENTENCE>"
NO_NEXT = "<NO NEXT SENTENCE>"


def parse_simple(value: Any) -> str:
    """Convert the raw list-like simple field into the unchanged target text."""
    if isinstance(value, list):
        return " ".join(str(item).strip() for item in value if str(item).strip())
    if pd.isna(value):
        return ""

    text = str(value).strip()
    if not text:
        return ""
    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        return text
    if isinstance(parsed, list):
        return " ".join(str(item).strip() for item in parsed if str(item).strip())
    return "" if parsed is None else str(parsed).strip()


def format_context(previous: str, current: str, next_sentence: str) -> str:
    return f"""Previous sentence:
{previous}

Current sentence:
{current}

Next sentence:
{next_sentence}

Simplify ONLY the current sentence."""


def load_context_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing {split_name} file: {path}")

    raw_df = pd.read_csv(path)
    missing_columns = [column for column in REQUIRED_COLUMNS if column not in raw_df.columns]
    if missing_columns:
        raise ValueError(f"{split_name} is missing columns: {missing_columns}")

    raw_df = raw_df.copy()
    raw_df["complex"] = raw_df["complex"].fillna("").astype(str).str.strip()
    raw_df["label"] = raw_df["label"].fillna("").astype(str).str.strip()
    raw_df["simple_parsed"] = raw_df["simple"].apply(parse_simple)

    if raw_df.duplicated(["pair_id", "sent_id"]).any():
        raise ValueError(f"{split_name} contains duplicate pair_id/sent_id keys.")

    sentence_lookup = raw_df.set_index(["pair_id", "sent_id"])["complex"].to_dict()
    sentence_keys = set(sentence_lookup)

    examples = raw_df[raw_df["label"].isin(KEEP_LABELS)].copy()
    examples = examples[examples["complex"].ne("") & examples["simple_parsed"].ne("")].copy()
    examples["simple"] = examples["simple_parsed"]
    examples["target_before_context"] = examples["simple"]

    def neighbour(row: pd.Series, offset: int, missing_value: str) -> str:
        key = (row["pair_id"], row["sent_id"] + offset)
        return sentence_lookup.get(key, missing_value)

    examples["previous_sentence"] = examples.apply(
        lambda row: neighbour(row, -1, NO_PREVIOUS), axis=1
    )
    examples["next_sentence"] = examples.apply(
        lambda row: neighbour(row, 1, NO_NEXT), axis=1
    )
    examples["previous_pair_id"] = examples.apply(
        lambda row: row["pair_id"]
        if (row["pair_id"], row["sent_id"] - 1) in sentence_keys
        else None,
        axis=1,
    )
    examples["next_pair_id"] = examples.apply(
        lambda row: row["pair_id"]
        if (row["pair_id"], row["sent_id"] + 1) in sentence_keys
        else None,
        axis=1,
    )

    previous_ok = examples["previous_pair_id"].isna() | examples["previous_pair_id"].eq(examples["pair_id"])
    next_ok = examples["next_pair_id"].isna() | examples["next_pair_id"].eq(examples["pair_id"])
    assert previous_ok.all(), f"{split_name}: previous sentence crossed a pair_id boundary."
    assert next_ok.all(), f"{split_name}: next sentence crossed a pair_id boundary."
    assert examples["simple"].equals(examples["target_before_context"]), (
        f"{split_name}: target text changed while context was reconstructed."
    )

    examples["context_input"] = examples.apply(
        lambda row: format_context(
            row["previous_sentence"], row["complex"], row["next_sentence"]
        ),
        axis=1,
    )

    output_columns = [
        "pair_id", "para_id", "sent_id", "label", "complex", "simple",
        "previous_sentence", "next_sentence", "context_input",
    ]
    return examples[output_columns].reset_index(drop=True)


train_df = load_context_split(TRAIN_PATH, "train")
val_df = load_context_split(VAL_PATH, "validation")
test_df = load_context_split(TEST_PATH, "test")

print(f"Loaded train: {len(train_df):,} rows from {TRAIN_PATH.relative_to(PROJECT_ROOT)}")
print(f"Loaded validation: {len(val_df):,} rows from {VAL_PATH.relative_to(PROJECT_ROOT)}")
print(f"Loaded test: {len(test_df):,} rows from {TEST_PATH.relative_to(PROJECT_ROOT)}")
print("Context boundary checks passed for all splits.")
print("Target preservation checks passed for all splits.")


## 3.Data Inspection



In [ ]:
context_columns = [
    "pair_id", "para_id", "sent_id", "previous_sentence", "complex",
    "next_sentence", "simple",
]
display(train_df[context_columns].sample(n=5, random_state=SEED))

def word_count(text: str) -> int:
    return len(str(text).split())

length_frames = []
for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    length_frames.append(
        pd.DataFrame(
            {
                "split": split_name,
                "context_words": df["context_input"].map(word_count),
                "complex_words": df["complex"].map(word_count),
                "simple_words": df["simple"].map(word_count),
            }
        )
    )
length_df = pd.concat(length_frames, ignore_index=True)

length_stats = (
    length_df.groupby("split")[["context_words", "complex_words", "simple_words"]]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(2)
)
display(length_stats)

print("Train label distribution:")
display(train_df["label"].value_counts(dropna=False).rename_axis("label").reset_index(name="count"))


## 3.Tokenization


In [ ]:
def load_tokenizer(model_candidates: list[str]) -> tuple[Any, str]:
    errors = []
    for candidate in model_candidates:
        try:
            loaded_tokenizer = AutoTokenizer.from_pretrained(candidate)
            print(f"Loaded tokenizer: {candidate}")
            return loaded_tokenizer, candidate
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
            print(f"Could not load tokenizer {candidate}: {exc}")
    raise RuntimeError("Could not load any tokenizer candidate.\n" + "\n".join(errors))


tokenizer, RESOLVED_MODEL_NAME = load_tokenizer(MODEL_CANDIDATES)

INITIAL_MAX_SOURCE_LENGTH = 256
EXTENDED_MAX_SOURCE_LENGTH = 384
TRUNCATION_THRESHOLD_PERCENT = 5.0
max_target_length = 128


def token_lengths(texts: list[str], batch_size: int = 256) -> list[int]:
    lengths = []
    for start in range(0, len(texts), batch_size):
        encoded = tokenizer(
            texts[start : start + batch_size],
            truncation=False,
            padding=False,
            add_special_tokens=True,
        )["input_ids"]
        lengths.extend(len(token_ids) for token_ids in encoded)
    return lengths


truncation_rows = []
all_initially_truncated = 0
all_examples = 0
for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    lengths = token_lengths(df["context_input"].tolist())
    truncated_256 = sum(length > INITIAL_MAX_SOURCE_LENGTH for length in lengths)
    truncated_384 = sum(length > EXTENDED_MAX_SOURCE_LENGTH for length in lengths)
    all_initially_truncated += truncated_256
    all_examples += len(lengths)
    truncation_rows.append(
        {
            "split": split_name,
            "examples": len(lengths),
            "truncated_at_256": truncated_256,
            "percent_at_256": 100 * truncated_256 / max(len(lengths), 1),
            "truncated_at_384": truncated_384,
            "percent_at_384": 100 * truncated_384 / max(len(lengths), 1),
        }
    )

truncation_df = pd.DataFrame(truncation_rows)
display(truncation_df.round(2))

overall_truncation_percent = 100 * all_initially_truncated / max(all_examples, 1)
max_source_length = (
    EXTENDED_MAX_SOURCE_LENGTH
    if overall_truncation_percent > TRUNCATION_THRESHOLD_PERCENT
    else INITIAL_MAX_SOURCE_LENGTH
)
print(f"Overall truncation at 256: {overall_truncation_percent:.2f}%")
print(f"Selected max_source_length = {max_source_length}")
print(f"max_target_length = {max_target_length}")


def build_prompt(context_input: str) -> str:
    return str(context_input).strip()


def preprocess_examples(examples: dict[str, list[Any]]) -> dict[str, Any]:
    inputs = [build_prompt(text) for text in examples["context_input"]]
    targets = [str(text).strip() for text in examples["simple"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_source_length,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
    )["input_ids"]

    labels = [
        [(token_id if token_id != tokenizer.pad_token_id else -100) for token_id in label]
        for label in labels
    ]

    model_inputs["labels"] = labels
    return model_inputs


sample_prompt = build_prompt(train_df.loc[0, "context_input"])
print(sample_prompt)
print("Target:", train_df.loc[0, "simple"])


## 4.Dataset Creation



In [ ]:
train_dataset_raw = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset_raw = Dataset.from_pandas(val_df, preserve_index=False)
test_dataset_raw = Dataset.from_pandas(test_df, preserve_index=False)

remove_columns = train_dataset_raw.column_names
train_dataset = train_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)
val_dataset = val_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)
test_dataset = test_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)

# Keep Python lists here. DataCollatorForSeq2Seq performs dynamic padding and tensor conversion.
# Pre-formatting as torch can trigger slow list-of-ndarray tensor warnings for labels.

print(train_dataset)
print(val_dataset)
print(test_dataset)

In [ ]:
def inspect_tokenized_labels(dataset: Dataset, name: str, n: int = 2) -> None:
    print(f"{name} label sanity check")
    for idx in range(min(n, len(dataset))):
        labels = dataset[idx]["labels"]
        if hasattr(labels, "tolist"):
            labels = labels.tolist()
        active_labels = [token_id for token_id in labels if token_id != -100]
        print(f"  example {idx}: active target tokens = {len(active_labels)}")
        print("  decoded target:", tokenizer.decode(active_labels, skip_special_tokens=True))
    empty_count = 0
    for row in dataset:
        labels = row["labels"]
        if hasattr(labels, "tolist"):
            labels = labels.tolist()
        if not any(token_id != -100 for token_id in labels):
            empty_count += 1
    print(f"  empty targets: {empty_count} / {len(dataset)}")
    if empty_count:
        raise ValueError(f"{name} has empty tokenized targets; check the simple column and preprocessing.")

inspect_tokenized_labels(train_dataset, "train")
inspect_tokenized_labels(val_dataset, "validation")

## 5.Model Loading



In [ ]:
CRITICAL_MISSING_KEYS = {
    "model.encoder.embed_tokens.weight",
    "model.decoder.embed_tokens.weight",
    "lm_head.weight",
}


def load_seq2seq_model(model_candidates: list[str], resolved_tokenizer_model: str) -> tuple[Any, str]:
    ordered_candidates = [resolved_tokenizer_model] + [
        candidate for candidate in model_candidates if candidate != resolved_tokenizer_model
    ]
    errors = []
    for candidate in ordered_candidates:
        try:
            model, loading_info = AutoModelForSeq2SeqLM.from_pretrained(
                candidate,
                output_loading_info=True,
            )
            missing_keys = set(loading_info.get("missing_keys", []))
            critical_missing = sorted(missing_keys & CRITICAL_MISSING_KEYS)
            if critical_missing:
                del model
                gc.collect()
                message = f"critical missing keys: {critical_missing}"
                errors.append(f"{candidate}: {message}")
                print(f"Skipping model {candidate}: {message}")
                continue
            unexpected_keys = loading_info.get("unexpected_keys", [])
            if unexpected_keys:
                print(f"Model {candidate} has unexpected keys: {unexpected_keys[:5]}")
            print(f"Loaded model: {candidate}")
            return model, candidate
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
            print(f"Could not load model {candidate}: {exc}")
    raise RuntimeError("Could not load any model candidate without critical missing keys.\n" + "\n".join(errors))

model, RESOLVED_MODEL_NAME = load_seq2seq_model(MODEL_CANDIDATES, RESOLVED_MODEL_NAME)
model.to(device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

print(f"Resolved model: {RESOLVED_MODEL_NAME}")
print(f"Model loaded on: {next(model.parameters()).device}")

## 6.Training


In [ ]:
def build_training_args() -> Seq2SeqTrainingArguments:
    base_kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=5,
        learning_rate=3e-5,
        weight_decay=0.01,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        predict_with_generate=True,
        fp16=fp16_enabled,
        bf16=bf16_enabled,
        logging_nan_inf_filter=False,
        report_to="none",
        seed=SEED,
    )
    try:
        return Seq2SeqTrainingArguments(
            evaluation_strategy="epoch",
            **base_kwargs,
        )
    except TypeError:
        return Seq2SeqTrainingArguments(
            eval_strategy="epoch",
            **base_kwargs,
        )

training_args = build_training_args()

def build_trainer() -> Seq2SeqTrainer:
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    try:
        return Seq2SeqTrainer(processing_class=tokenizer, **trainer_kwargs)
    except TypeError:
        return Seq2SeqTrainer(tokenizer=tokenizer, **trainer_kwargs)

trainer = build_trainer()
trainer.train()
trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))
print(f"Saved best model to: {BEST_MODEL_DIR.relative_to(PROJECT_ROOT)}")

### Training Log



In [ ]:
training_log_df = pd.DataFrame(trainer.state.log_history)
display(training_log_df)

epoch_eval_df = training_log_df[training_log_df["eval_loss"].notna()].copy()
if len(epoch_eval_df):
    display(epoch_eval_df[["epoch", "step", "eval_loss", "eval_runtime"]])
else:
    print("No eval_loss rows found in trainer log history.")

## 7.Predictions




In [ ]:
GENERATION_CONFIG = {
    "max_new_tokens": max_target_length,
    "num_beams": 4,
    "length_penalty": 0.9,
    "no_repeat_ngram_size": 3,
    "early_stopping": True,
}

print("Generation config:", GENERATION_CONFIG)

def clean_prediction(text: str) -> str:
    """Remove prompt echoes and generation boilerplate from decoded text."""
    text = re.sub(r"\s+", " ", str(text).strip())
    if not text:
        return ""

    prompt_markers = [
        "Simplified sentence:",
        "Simplify ONLY the current sentence.",
        "Current sentence:",
        "Next sentence:",
        "Previous sentence:",
    ]
    for marker in prompt_markers:
        if marker in text:
            text = text.split(marker)[-1].strip()

    prefixes = ["Simplified:", "Answer:", "Prediction:"]
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()

    return re.sub(r"\s+", " ", text).strip()


def generate_batch(context_inputs: list[str]) -> list[str]:
    input_texts = [build_prompt(text) for text in context_inputs]

    inputs = tokenizer(
        input_texts,
        max_length=max_source_length,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **GENERATION_CONFIG,
        )

    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    return [clean_prediction(text) for text in decoded]


def generate_predictions(df: pd.DataFrame, batch_size: int = 8) -> pd.DataFrame:
    model.eval()
    predictions = []
    context_inputs = df["context_input"].fillna("").astype(str).tolist()

    for start in tqdm(range(0, len(context_inputs), batch_size), desc="Generating"):
        batch_inputs = context_inputs[start : start + batch_size]
        try:
            batch_predictions = generate_batch(batch_inputs)
        except Exception as exc:
            print(f"Generation failed for rows {start}-{start + len(batch_inputs) - 1}: {exc}")
            batch_predictions = [""] * len(batch_inputs)

        predictions.extend(batch_predictions)

    output_df = df[["pair_id", "sent_id", "label", "complex", "simple"]].copy()
    output_df["prediction"] = predictions
    return output_df


prediction_df = generate_predictions(test_df, batch_size=8)
prediction_df.to_csv(PREDICTION_PATH, index=False)

print(f"Saved predictions to: {PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
display(prediction_df.head())


## 8.Evaluation

The metrics mirror the FLAN-T5 notebook. SARI evaluates simplification edits, BLEU is reported with sacreBLEU on a `0-100` scale for comparability with the E1 result, and BERTScore measures semantic similarity.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import compute_metrics

metrics_summary = compute_metrics(prediction_df)
display(metrics_summary)


## 9.Analysis


In [ ]:
example_columns = ["complex", "simple", "prediction"]
qualitative_examples = prediction_df[example_columns].sample(
    n=min(20, len(prediction_df)),
    random_state=SEED,
)
display(qualitative_examples)

## 10.Context Comparison

The two BioBART systems are compared with the shared functions from `src/evaluation.py`. The no-context row is recomputed from its saved predictions when that file is available, so both rows use the same metric implementation and scale.


In [ ]:
def metric_value(summary: pd.DataFrame, metric: str) -> float:
    values = summary.loc[summary["metric"].eq(metric), "score"].tolist()
    return float(values[0]) if values else float("nan")


NO_CONTEXT_PREDICTION_PATH = RESULTS_DIR / "biobart_sentence_no_context_predictions.csv"
if NO_CONTEXT_PREDICTION_PATH.exists():
    no_context_prediction_df = pd.read_csv(NO_CONTEXT_PREDICTION_PATH)
    key_columns = ["pair_id", "sent_id", "complex", "simple"]
    if len(no_context_prediction_df) != len(prediction_df):
        raise ValueError("No-context and context prediction files have different row counts.")
    for column in key_columns:
        left = no_context_prediction_df[column].fillna("").astype(str).reset_index(drop=True)
        right = prediction_df[column].fillna("").astype(str).reset_index(drop=True)
        if not left.equals(right):
            raise ValueError(f"Prediction files are not aligned on {column}.")
    no_context_metrics = compute_metrics(no_context_prediction_df)
else:
    print(f"No-context predictions not found: {NO_CONTEXT_PREDICTION_PATH}")
    no_context_metrics = pd.DataFrame(
        {
            "metric": ["SARI", "BLEU", "BERTScore F1"],
            "score": [31.86, 30.91, 0.932],
        }
    )

comparison_df = pd.DataFrame(
    [
        {
            "Model": "BioBART no context",
            "SARI": metric_value(no_context_metrics, "SARI"),
            "BLEU": metric_value(no_context_metrics, "BLEU"),
            "BERTScore F1": metric_value(no_context_metrics, "BERTScore F1"),
        },
        {
            "Model": "BioBART with context",
            "SARI": metric_value(metrics_summary, "SARI"),
            "BLEU": metric_value(metrics_summary, "BLEU"),
            "BERTScore F1": metric_value(metrics_summary, "BERTScore F1"),
        },
    ]
)
display(comparison_df)
